<a href="https://colab.research.google.com/github/MalakSalehh/Thesis-datasets/blob/main/GPT4o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
!pip install -q openai pandas numpy scipy scikit-learn

In [32]:
import os
import re
import json
import time
import numpy as np
import pandas as pd

from openai import OpenAI
from sklearn.metrics import cohen_kappa_score, mean_absolute_error
from scipy.stats import pearsonr

In [6]:


client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [33]:
##response function
def get_response_enhanced(prompt, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You are an expert essay grader. "
                            "Follow the rubric and calibration exactly. "
                            "Return valid JSON only."
                        )
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=0.05,
                max_tokens=180
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(3 * (attempt + 1))

    return None

In [34]:
##load dataset
import pandas as pd

url = "https://raw.githubusercontent.com/MalakSalehh/Thesis-datasets/main/training_set_rel3.tsv"
df = pd.read_csv(url, sep="\t", encoding="latin1")

print(df.shape)
df.head()


(12976, 28)


,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [35]:

set1 = df[df["essay_set"] == 1].copy()
set1 = set1[["essay_id", "essay", "domain1_score"]]

set1["essay"] = set1["essay"].astype(str).str.replace("\n", " ", regex=False).str.strip()

print(set1.shape)
set1.head()

(1783, 3)


,essay_id,essay,domain1_score
0,1,"Dear local newspaper, I think effects computer...",8
1,2,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,3,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,4,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,5,"Dear @LOCATION1, I know having computers has a...",8


In [36]:
def categorize(score):
    if score <= 5:
        return "Low"
    elif score <= 8:
        return "Medium"
    else:
        return "High"

set1["score_category"] = set1["domain1_score"].apply(categorize)

set1["domain1_score"].value_counts().sort_index()

,count
domain1_score,
2,10
3,1
4,17
5,17
6,110
7,135
8,687
9,334
10,316


In [37]:
RANDOM_STATE = 42

calibration_pool = (
    set1.groupby("domain1_score", group_keys=False)
    .apply(lambda x: x.sample(min(1, len(x)), random_state=RANDOM_STATE))
    .reset_index(drop=True)
)

calibration = calibration_pool.sample(6, random_state=RANDOM_STATE).reset_index(drop=True)

calibration

/tmp/ipykernel_8115/73661984.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(1, len(x)), random_state=RANDOM_STATE))


,essay_id,essay,domain1_score,score_category
0,1277,"Dear @CAPS1 @CAPS2, I am writing you this lett...",7,Medium
1,1592,The computers are cool. Do you now I werpsite ...,2,Low
2,1304,Would you like to discover new things and expl...,11,High
3,953,"Dear @CAPS1 of @LOCATION1, @CAPS2 the world ha...",12,High
4,19,I aegre waf the evansmant ov tnachnolage. The ...,4,Low
5,22,Dear local Newspaper @CAPS1 a take all your co...,3,Low


In [38]:
## removing calibration from dataset and sample 30 test essays
calibration_ids = calibration["essay_id"].tolist()

pool = set1[~set1["essay_id"].isin(calibration_ids)].copy().reset_index(drop=True)

demo_30 = pool.sample(30, random_state=RANDOM_STATE).reset_index(drop=True)

print("Calibration:", calibration.shape)
print("Demo 30:", demo_30.shape)
demo_30.head()

Calibration: (6, 4)
Demo 30: (30, 4)


,essay_id,essay,domain1_score,score_category
0,68,"Dear Local Newspaper @CAPS1, I am writing to y...",9,High
1,946,According to the @ORGANIZATION1 govermant @PER...,8,Medium
2,837,"Dear, @ORGANIZATION1 the big rectangular shape...",8,Medium
3,1710,Wouldn't you agree that computers have a great...,9,High
4,804,"Dear Local Newspaper, The effects computers ha...",8,Medium


In [39]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores.
Final score range: 2–12.
"""

In [16]:
##calibration format with ids
def format_calibration_examples_with_id(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [17]:
##building the prompt
def build_mts_prompt_enhanced(essay_id, essay_text, rubric_text, calibration_text_with_id):
    return f"""

{rubric_text}
Note: Final domain1_score = rater1(1-6) + rater2(1-6), so final scores range from 2 to 12.

Content = relevance of ideas, argument development, supporting details
Organization = structure, coherence, logical flow, paragraphing
Language = grammar, vocabulary, clarity, sentence control

{calibration_text_with_id}

Pattern:
- Low (2-5) = weak development and weak organization
- Medium (6-8) = adequate support and reasonable structure
- High (9-12) = strong reasoning, development, and structure

Essay ID: {essay_id}

Essay:
{essay_text}

1. Find the closest calibration example by overall writing quality.
2. Judge Content, Organization, and Language separately using the rubric.
3. Use the full trait range from 1 to 6 when justified.
4. Strong essays should receive 5s or 6s on multiple traits.
5. Return valid JSON only.

{{
  "essay_id": {essay_id},
  "content": X,
  "organization": X,
  "language": X,
  "reasoning": "2 short sentences"
}}

"""

In [41]:
##JSON parser
def parse_json_output(text):
    if not isinstance(text, str):
        return {}

    try:
        json_match = re.search(r"\{.*\}", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except Exception:
        pass

    parsed = {}

    essay_id_match = re.search(r'"?essay_id"?\s*:\s*(\d+)', text, re.IGNORECASE)
    content_match = re.search(r'"?content"?\s*:\s*(\d+)', text, re.IGNORECASE)
    org_match = re.search(r'"?organization"?\s*:\s*(\d+)', text, re.IGNORECASE)
    lang_match = re.search(r'"?language"?\s*:\s*(\d+)', text, re.IGNORECASE)
    reasoning_match = re.search(r'"?reasoning"?\s*:\s*"([^"]+)"', text, re.IGNORECASE)

    if essay_id_match:
        parsed["essay_id"] = int(essay_id_match.group(1))
    if content_match:
        parsed["content"] = int(content_match.group(1))
    if org_match:
        parsed["organization"] = int(org_match.group(1))
    if lang_match:
        parsed["language"] = int(lang_match.group(1))
    if reasoning_match:
        parsed["reasoning"] = reasoning_match.group(1)

    return parsed

In [57]:
def compute_mts_score(content, organization, language):
    if None in [content, organization, language]:
        return None, None

    # simulate two raters
    rater1 = content
    rater2 = int(round((organization + language) / 2))

    final_score = rater1 + rater2
    final_score = max(2, min(12, final_score))

    if final_score <= 5:
        category = "Low"
    elif final_score <= 8:
        category = "Medium"
    else:
        category = "High"

    return final_score, category

In [43]:
##run on 30 essays
results = []

for i, row in demo_30.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]
    human_category = row["score_category"]

    prompt = build_mts_prompt_enhanced(
        essay_id=essay_id,
        essay_text=essay_text,
        rubric_text=rubric_text,
        calibration_text_with_id=calibration_text_with_id
    )

    output = get_response_enhanced(prompt)
    parsed = parse_json_output(output)

    content_score = clean_trait_score(parsed.get("content"))
    organization_score = clean_trait_score(parsed.get("organization"))
    language_score = clean_trait_score(parsed.get("language"))
    reasoning = parsed.get("reasoning")

    predicted_score, predicted_category = compute_mts_score(
        content_score,
        organization_score,
        language_score
    )

    results.append({
        "essay_id": essay_id,
        "essay": essay_text,
        "human_score": human_score,
        "human_category": human_category,
        "content_score": content_score,
        "organization_score": organization_score,
        "language_score": language_score,
        "predicted_score": predicted_score,
        "predicted_category": predicted_category,
        "reasoning": reasoning,
        "raw_output": output
    })

    print(f"Done essay {essay_id}")

    if (i + 1) % 5 == 0:
        pd.DataFrame(results).to_csv("gpt4o_partial_results.csv", index=False)
        print(f"Saved progress after {i+1} essays")

    time.sleep(1)

Done essay 68
Done essay 946
Done essay 837
Done essay 1710
Done essay 804
Saved progress after 5 essays
Done essay 1541
Done essay 1042
Done essay 242
Done essay 1211
Done essay 1487
Saved progress after 10 essays
Done essay 941
Done essay 1126
Done essay 1566
Done essay 1730
Done essay 1073
Saved progress after 15 essays
Done essay 922
Done essay 112
Done essay 1588
Done essay 559
Done essay 886
Saved progress after 20 essays
Done essay 356
Done essay 403
Done essay 1521
Done essay 1404
Done essay 326
Saved progress after 25 essays
Done essay 537
Done essay 1353
Done essay 699
Done essay 1691
Done essay 738
Saved progress after 30 essays


In [59]:
results_df = pd.DataFrame(results)
results_df.to_csv("gpt4o_enhanced_mts_results.csv", index=False)

results_df[[
    "essay_id",
    "essay",
    "human_score",
    "human_category",
    "content_score",
    "organization_score",
    "language_score",
    "predicted_score",
    "predicted_category"
]]

,essay_id,essay,human_score,human_category,content_score,organization_score,language_score,predicted_score,predicted_category
0,68,"Dear Local Newspaper @CAPS1, I am writing to y...",9,High,4,4,4,8,Medium
1,946,According to the @ORGANIZATION1 govermant @PER...,8,Medium,4,3,3,7,Medium
2,837,"Dear, @ORGANIZATION1 the big rectangular shape...",8,Medium,3,2,2,5,Low
3,1710,Wouldn't you agree that computers have a great...,9,High,4,3,3,7,Medium
4,804,"Dear Local Newspaper, The effects computers ha...",8,Medium,3,3,3,6,Medium
5,1541,"Dear NewsPaper, people are spending too much t...",8,Medium,3,3,3,6,Medium
6,1042,"Dear @ORGANIZATION1, I took a poll of @NUM1 pe...",8,Medium,5,5,4,9,High
7,242,"Dear Newspaper, In our town, more people are o...",9,High,4,4,4,8,Medium
8,1211,Dear editor: @CAPS1 you know that @PERCENT1 of...,9,High,5,5,4,9,High
9,1487,Studies show that @PERCENT1 of people that are...,8,Medium,3,3,3,6,Medium


In [60]:
results_df["predicted_score"], results_df["predicted_category"] = zip(
    *results_df.apply(
        lambda row: compute_mts_score(
            row["content_score"],
            row["organization_score"],
            row["language_score"]
        ),
        axis=1
    )
)

In [61]:
clean_df = results_df.dropna(subset=["predicted_score"]).copy()

qwk = cohen_kappa_score(
    clean_df["human_score"],
    clean_df["predicted_score"],
    weights="quadratic",
    labels=list(range(2, 13))
)

pcc, _ = pearsonr(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

mae = mean_absolute_error(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

print("GPT-4o Enhanced MTS QWK:", qwk)
print("GPT-4o Enhanced MTS PCC:", pcc)
print("GPT-4o Enhanced MTS MAE:", mae)

GPT-4o Enhanced MTS QWK: 0.2753623188405797
GPT-4o Enhanced MTS PCC: 0.4380858271151807
GPT-4o Enhanced MTS MAE: 1.7333333333333334


In [62]:
results_df["content_score"].value_counts().sort_index()
results_df["organization_score"].value_counts().sort_index()
results_df["language_score"].value_counts().sort_index()

,count
language_score,
2,4
3,16
4,7
5,3


In [63]:
def build_cj_prompt(essay_id, essay_text, rubric_text, calibration_text_with_id):
    return f"""
{rubric_text}

You are an expert essay grader.

Below are calibration examples with known human scores.

{calibration_text_with_id}

Now evaluate the following essay.

Essay ID: {essay_id}

Essay:
{essay_text}

Step 1:
Compare this essay to the calibration examples.
- Identify which example it is most similar to in overall quality.
- State whether it is better, worse, or similar.

Step 2:
Based on this comparison, assign:
- Content score (1–6)
- Organization score (1–6)
- Language score (1–6)

Step 3:
Ensure you use the FULL score range (1–6).
High-quality essays MUST receive 5 or 6 where appropriate.

Step 4:
Return valid JSON only:

{{
  "essay_id": {essay_id},
  "closest_example": X,
  "comparison": "better / worse / similar",
  "content": X,
  "organization": X,
  "language": X,
  "reasoning": "2 short sentences explaining comparison"
}}
"""

In [64]:
results = []

for i, row in demo_30.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]
    human_category = row["score_category"]

    prompt = build_cj_prompt(
        essay_id=essay_id,
        essay_text=essay_text,
        rubric_text=rubric_text,
        calibration_text_with_id=calibration_text_with_id
    )

    output = get_response_enhanced(prompt)
    parsed = parse_json_output(output)

    content = clean_trait_score(parsed.get("content"))
    organization = clean_trait_score(parsed.get("organization"))
    language = clean_trait_score(parsed.get("language"))

    predicted_score, predicted_category = compute_mts_score(
        content, organization, language
    )

    results.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "human_category": human_category,
        "content_score": content,
        "organization_score": organization,
        "language_score": language,
        "predicted_score": predicted_score,
        "predicted_category": predicted_category,
        "raw_output": output
    })

    print(f"Done {essay_id}")

    if (i + 1) % 5 == 0:
        pd.DataFrame(results).to_csv("cj_partial.csv", index=False)

    time.sleep(1)

Done 68
Done 946
Done 837
Done 1710
Done 804
Done 1541
Done 1042
Done 242
Done 1211
Done 1487
Done 941
Done 1126
Done 1566
Done 1730
Done 1073
Done 922
Done 112
Done 1588
Done 559
Done 886
Done 356
Done 403
Done 1521
Done 1404
Done 326
Done 537
Done 1353
Done 699
Done 1691
Done 738


In [65]:
results_df = pd.DataFrame(results)

results_df[[
    "essay_id",
    "human_score",
    "human_category",
    "content_score",
    "organization_score",
    "language_score",
    "predicted_score",
    "predicted_category"
]]

,essay_id,human_score,human_category,content_score,organization_score,language_score,predicted_score,predicted_category
0,68,9,High,4,4,4,8,Medium
1,946,8,Medium,3,3,2,5,Low
2,837,8,Medium,3,3,2,5,Low
3,1710,9,High,4,4,3,8,Medium
4,804,8,Medium,4,4,3,8,Medium
5,1541,8,Medium,4,4,4,8,Medium
6,1042,8,Medium,4,4,4,8,Medium
7,242,9,High,4,4,4,8,Medium
8,1211,9,High,5,5,5,10,High
9,1487,8,Medium,3,3,3,6,Medium


In [66]:
clean_df = results_df.dropna(subset=["predicted_score"])

qwk = cohen_kappa_score(
    clean_df["human_score"],
    clean_df["predicted_score"],
    weights="quadratic",
    labels=list(range(2, 13))
)

pcc, _ = pearsonr(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

mae = mean_absolute_error(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

print("CJ QWK:", qwk)
print("CJ PCC:", pcc)
print("CJ MAE:", mae)

CJ QWK: 0.3419788664745437
CJ PCC: 0.46175548068680927
CJ MAE: 1.5


In [73]:
##Calibration set increase
RANDOM_STATE = 42

calibration = (
    set1.groupby("domain1_score", group_keys=False)
    .apply(lambda x: x.sample(1, random_state=RANDOM_STATE))
    .reset_index(drop=True)
    .sort_values("domain1_score")
    .reset_index(drop=True)
)

print("Calibration size:", calibration.shape)
calibration[["essay_id", "domain1_score", "score_category"]]

Calibration size: (11, 4)


/tmp/ipykernel_8115/2016771049.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(1, random_state=RANDOM_STATE))


,essay_id,domain1_score,score_category
0,1592,2,Low
1,22,3,Low
2,19,4,Low
3,50,5,Low
4,1222,6,Medium
5,1277,7,Medium
6,753,8,Medium
7,157,9,High
8,954,10,High
9,1304,11,High


In [74]:
calibration_ids = calibration["essay_id"].tolist()

pool = set1[~set1["essay_id"].isin(calibration_ids)].copy().reset_index(drop=True)

demo_30 = pool.sample(30, random_state=RANDOM_STATE).reset_index(drop=True)

print("Calibration:", calibration.shape)
print("Demo 30:", demo_30.shape)

Calibration: (11, 4)
Demo 30: (30, 4)


In [75]:
def format_calibration_examples_with_id(calibration_df):
    examples = []
    for i, row in enumerate(calibration_df.itertuples(index=False), start=1):
        examples.append(f"""
Calibration Example {i}

Essay ID: {row.essay_id}

Essay:
{row.essay}

Human Score: {row.domain1_score}
Category: {row.score_category}
""")
    return "\n".join(examples)

calibration_text_with_id = format_calibration_examples_with_id(calibration)

In [76]:
rubric_text = """
Essay Prompt:
Write a letter to your local newspaper stating your opinion on the effects computers have on people and persuade readers to agree with your position.

Rubric:
1 – Undeveloped response with minimal support.
2 – Underdeveloped response with weak reasoning.
3 – Minimally developed argument with limited support.
4 – Adequate argument with some supporting details.
5 – Developed argument with clear reasoning and organization.
6 – Strong persuasive response with well-developed ideas.

Scoring rule:
Two raters assign scores from 1–6.
Final score = sum of both scores (2–12).
"""

In [77]:
def build_cj_direct_prompt(essay_id, essay_text, rubric_text, calibration_text_with_id):
    return f"""
{rubric_text}

You are an expert essay grader.

Below are calibration examples with known human scores.

{calibration_text_with_id}

Now evaluate this essay.

Essay ID: {essay_id}

Essay:
{essay_text}

Instructions:
1. Compare this essay to the calibration examples.
2. Identify the closest example in quality.
3. Decide if it is better, worse, or similar.
4. Assign a FINAL SCORE (2–12) directly.
5. Use the FULL range when justified.

Return JSON only:

{{
  "essay_id": {essay_id},
  "closest_example": X,
  "comparison": "better / worse / similar",
  "final_score": X,
  "reasoning": "short explanation"
}}
"""

In [78]:
def parse_cj_direct_output(text):
    if not isinstance(text, str):
        return {}

    try:
        json_match = re.search(r"\{.*\}", text, re.DOTALL)
        if json_match:
            return json.loads(json_match.group())
    except:
        pass

    return {}

In [79]:
results = []

for i, row in demo_30.iterrows():
    essay_id = row["essay_id"]
    essay_text = row["essay"]
    human_score = row["domain1_score"]
    human_category = row["score_category"]

    prompt = build_cj_direct_prompt(
        essay_id,
        essay_text,
        rubric_text,
        calibration_text_with_id
    )

    output = get_response_enhanced(prompt)
    parsed = parse_cj_direct_output(output)

    predicted_score = parsed.get("final_score")

    if predicted_score is not None:
        if predicted_score <= 5:
            predicted_category = "Low"
        elif predicted_score <= 8:
            predicted_category = "Medium"
        else:
            predicted_category = "High"
    else:
        predicted_category = None

    results.append({
        "essay_id": essay_id,
        "human_score": human_score,
        "human_category": human_category,
        "closest_example": parsed.get("closest_example"),
        "comparison": parsed.get("comparison"),
        "predicted_score": predicted_score,
        "predicted_category": predicted_category,
        "reasoning": parsed.get("reasoning")
    })

    print(f"Done {essay_id}")

    if (i + 1) % 5 == 0:
        pd.DataFrame(results).to_csv("cj_partial.csv", index=False)

    time.sleep(1)

Done 1219
Done 1207
Done 800
Done 1668
Done 1180
Done 1046
Done 501
Done 1526
Done 1062
Done 1694
Done 215
Done 389
Done 1200
Done 918
Done 1663
Done 337
Done 275
Done 252
Done 568
Done 69
Done 772
Done 1367
Done 875
Done 1483
Done 217
Done 1578
Done 74
Done 357
Done 1469
Done 1650


In [83]:
results_df = pd.DataFrame(results)

results_df[[
    "essay_id",
    "human_score",
    "human_category",
    "predicted_score",
    "predicted_category"
]]

,essay_id,human_score,human_category,predicted_score,predicted_category
0,1219,8,Medium,8,Medium
1,1207,8,Medium,10,High
2,800,10,High,8,Medium
3,1668,9,High,8,Medium
4,1180,8,Medium,8,Medium
5,1046,9,High,4,Low
6,501,8,Medium,7,Medium
7,1526,10,High,8,Medium
8,1062,10,High,3,Low
9,1694,10,High,8,Medium


In [84]:
clean_df = results_df.dropna(subset=["predicted_score"])

qwk = cohen_kappa_score(
    clean_df["human_score"],
    clean_df["predicted_score"],
    weights="quadratic",
    labels=list(range(2, 13))
)

pcc, _ = pearsonr(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

mae = mean_absolute_error(
    clean_df["human_score"],
    clean_df["predicted_score"]
)

print("QWK:", qwk)
print("PCC:", pcc)
print("MAE:", mae)

QWK: 0.31921331316187584
PCC: 0.506217956258354
MAE: 1.8
